In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as pl
from joblib import dump, load
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import RidgeCV, Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

In [2]:
hours = np.array([1.5, 2, 3, 4, 5, 6, 7, 8, 10]).reshape(-1, 1)
profit = np.array([40, 50, 70, 90, 100, 110, 130, 145, 180]).reshape(-1, 1)

In [3]:
poly_con = PolynomialFeatures(degree=4, include_bias=False)
norm_con = Normalizer()

ploy_hours = poly_con.fit_transform(hours)
poly_hours_train, poly_hours_test, profit_train, profit_test = train_test_split(ploy_hours, profit, test_size=0.2, random_state=101)
norm_con.fit(poly_hours_train)
norm_poly_hours_train = norm_con.transform(poly_hours_train)
norm_poly_hours_test = norm_con.transform(poly_hours_test)


In [4]:
ridge = Ridge(alpha=5)
ridge.fit(norm_poly_hours_train, profit_train)
profit_predictions = ridge.predict(norm_poly_hours_test)
mae = mean_absolute_error(profit_test, profit_predictions)
rmse = root_mean_squared_error(profit_test, profit_predictions)
print(mae)
print(rmse)

36.366341417701555
36.98739144669071


In [5]:
alphas = np.arange(0.001, 1, 0.1)
ridge_cv = RidgeCV(alphas=alphas, scoring="neg_root_mean_squared_error")
ridge_cv.fit(norm_poly_hours_train, profit_train)
cv_profit_predictions = ridge_cv.predict(norm_poly_hours_test)
cv_mae = mean_absolute_error(profit_test, cv_profit_predictions)
cv_rmse = root_mean_squared_error(profit_test, cv_profit_predictions)
print(cv_mae)
print(cv_rmse)

2.6481629803539732
2.6712071817951326


In [6]:
ridge_cv.alpha_

0.001

In [7]:
ridge_cv.coef_

array([ 105.25372038,  201.16315443, -526.82683956,  -63.18176258])

In [8]:
norm_poly_hours = norm_con.fit_transform(ploy_hours)
final_ridge = Ridge(alpha=0.00001)
final_ridge.fit(norm_poly_hours, profit)
dump(final_ridge, 'final_ridge_model')
dump(norm_con, 'normalizer')
dump(poly_con, 'polynomial_converter')

['polynomial_converter']

In [9]:
norm = load('normalizer')
poly = load('polynomial_converter')
model = load('final_ridge_model')

new_data = np.array([20]).reshape(-1,1)
poly_new_data = poly.transform(new_data)
norm_poly_new_data = norm.transform(poly_new_data)
preds = model.predict(norm_poly_new_data)
preds

array([214.2569786])